In [0]:
import numpy as np, pandas as pd, os

rng = np.random.default_rng(42)
BASE = "/Volumes/Project_Lakehouse_telemetria_ELE/telemetria_bronze/landing/telemetria"
os.makedirs(BASE, exist_ok=True)

N_DEVICES, DAYS, READS_PER_DAY = 50, 7, 3000   # ~1,05 milhão de linhas
devices = [f"TRF-{i:03d}" for i in range(N_DEVICES)]
start = pd.Timestamp("2026-10-01")

for d in range(DAYS):
    day = start + pd.Timedelta(days=d)
    frames = []
    for dev in devices:
        ts = day + pd.to_timedelta(np.sort(rng.integers(0, 86400, READS_PER_DAY)), unit="s")
        hour = ts.hour.values
        load = 0.6 + 0.3 * np.sin((hour - 6) / 24 * 2 * np.pi)      # curva de carga diária
        frames.append(pd.DataFrame({
            "device_id": dev,
            "event_ts": ts,
            "tensao_v": 220 + rng.normal(0, 2, READS_PER_DAY),
            "corrente_a": 40 * load + rng.normal(0, 1.5, READS_PER_DAY),
            "temperatura_c": 45 + 25 * load + rng.normal(0, 1.2, READS_PER_DAY),
            "alarme": "OK",
        }))
    df = pd.concat(frames, ignore_index=True)
    n = len(df)

    # Falhas propositais (alvo da Fase 2)
    df.loc[df.sample(frac=0.010, random_state=d).index, "tensao_v"] = np.nan          # nulos
    df.loc[df.sample(frac=0.003, random_state=d+1).index, "temperatura_c"] += 60      # picos
    df.loc[df.temperatura_c > 100, "alarme"] = "SOBRETEMP"
    df.loc[df.sample(frac=0.002, random_state=d+2).index, "corrente_a"] = -999        # sentinela
    df = pd.concat([df, df.sample(frac=0.005, random_state=d+3)], ignore_index=True)  # duplicados
    df["event_ts"] = df["event_ts"].astype(str)

    df.sample(frac=1, random_state=d).to_csv(f"{BASE}/telemetria_{day:%Y%m%d}.csv", index=False)
    print(day.date(), len(df))